In [ ]:
import urllib.request

url = "https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip"

urllib.request.urlretrieve(url, "maestro.zip")

print("Download complete")

In [ ]:
import zipfile

with zipfile.ZipFile("maestro.zip", "r") as zip_ref:
    zip_ref.extractall("midi_data")

print("MIDI files extracted")

In [ ]:
import glob

midi_files = glob.glob("midi_data/**/*.midi", recursive=True)

print("Number of MIDI files:", len(midi_files))

In [ ]:
midi_files = midi_files[:100]

print("MIDI files selected:", len(midi_files))

In [ ]:
from mido import MidiFile

notes = []

for file in midi_files:
    try:
        midi = MidiFile(file)

        for track in midi.tracks:
            for message in track:
                if message.type == 'note_on' and message.velocity > 0:
                    notes.append(message.note)

    except Exception:
        pass

print("Total notes collected:", len(notes))

In [ ]:
!pip install torch mido numpy

In [ ]:
from mido import MidiFile

notes = []

for file in midi_files:
    try:
        midi = MidiFile(file)

        for track in midi.tracks:
            for message in track:
                if message.type == 'note_on' and message.velocity > 0:
                    notes.append(message.note)

    except Exception:
        pass

print("Total notes collected:", len(notes))

In [ ]:
sequence_length = 50

sequences = []
targets = []

for i in range(len(notes) - sequence_length):
    sequences.append(notes[i:i + sequence_length])
    targets.append(notes[i + sequence_length])

print("Number of sequences:", len(sequences))

In [ ]:
unique_notes = sorted(set(notes))

note_to_int = {note: i for i, note in enumerate(unique_notes)}
int_to_note = {i: note for i, note in enumerate(unique_notes)}

sequences = [[note_to_int[note] for note in seq] for seq in sequences]
targets = [note_to_int[note] for note in targets]

print("Unique notes:", len(unique_notes))

In [ ]:
import torch

X = torch.tensor(sequences, dtype=torch.long)
y = torch.tensor(targets, dtype=torch.long)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X, y)

batch_size = 128

loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True
)

print("Batches:", len(loader))

In [ ]:
import torch.nn as nn

class MusicLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()

        self.embedding = nn.Embedding(input_size, 64)

        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=128,
            batch_first=True
        )

        self.fc = nn.Linear(128, output_size)

    def forward(self, x):
        x = self.embedding(x)
        output, _ = self.lstm(x)
        output = output[:, -1, :]
        output = self.fc(output)

        return output


input_size = len(unique_notes)
hidden_size = 128
output_size = len(unique_notes)

model = MusicLSTM(input_size, hidden_size, output_size)

print(model)

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Training setup complete")

In [ ]:
torch.save(model.state_dict(), "music_lstm.pth")

print("Model saved successfully")

In [ ]:
import random

model.eval()

start_index = random.randint(0, len(X) - 1)

generated = X[start_index].tolist()

for _ in range(200):
    input_sequence = torch.tensor(
        [generated[-50:]],
        dtype=torch.long
    )

    with torch.no_grad():
        prediction = model(input_sequence)

    next_note = torch.argmax(prediction, dim=1).item()

    generated.append(next_note)

print("Generated notes:", len(generated))

In [ ]:
generated_notes = [
    int_to_note[i] for i in generated
]

print(generated_notes[:20])

In [ ]:
from mido import MidiFile, MidiTrack, Message

midi = MidiFile()
track = MidiTrack()
midi.tracks.append(track)

for note in generated_notes:
    track.append(
        Message('note_on', note=note, velocity=80, time=0)
    )

    track.append(
        Message('note_off', note=note, velocity=80, time=480)
    )

midi.save("generated_music.mid")

print("MIDI file created successfully")

In [ ]:
from google.colab import files

files.download("generated_music.mid")

In [ ]:
from mido import MidiFile, MidiTrack, Message, MetaMessage

midi = MidiFile(ticks_per_beat=480)
track = MidiTrack()
midi.tracks.append(track)

# Set tempo
track.append(MetaMessage('set_tempo', tempo=500000))

# Set piano instrument
track.append(Message('program_change', program=0))

for note in generated_notes:
    track.append(
        Message('note_on', note=int(note), velocity=80, time=0)
    )

    track.append(
        Message('note_off', note=int(note), velocity=80, time=480)
    )

track.append(MetaMessage('end_of_track'))

midi.save("generated_music_piano.mid")

print("New MIDI file created successfully")

In [ ]:
from google.colab import files

files.download("generated_music_piano.mid")

In [ ]:
!apt-get update -qq
!apt-get install -y fluidsynth

In [ ]:
!wget -q https://github.com/FluidSynth/fluidsynth/raw/master/sf2/VintageDreamsWaves-v2.0.sf2 -O piano.sf2

In [ ]:
!fluidsynth -ni piano.sf2 generated_music_piano.mid -F generated_music.wav -r 44100

In [ ]:
from IPython.display import Audio

Audio("generated_music.wav")

In [ ]:
!mkdir -p music_project

!cp generated_music_piano.mid music_project/
!cp generated_music.wav music_project/
!cp music_lstm.pth music_project/

print("Project files saved")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

drive_folder = "/content/drive/MyDrive/LSTM_Music_Generator"

os.makedirs(drive_folder, exist_ok=True)

!cp music_lstm.pth "$drive_folder/"
!cp generated_music_piano.mid "$drive_folder/"
!cp generated_music.wav "$drive_folder/"

print("Project files saved to Google Drive")